In [1]:
from sqlalchemy import create_engine
import psycopg2
import pandas as pd

In [2]:
username = 'postgres'
password = '1234'
host = 'localhost'
port = '5432'
database = 'uber_db'

In [3]:
conn = create_engine(f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}')

In [4]:
data = pd.read_csv("data.csv")

In [5]:
data.head(5)

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [6]:
data = data.drop(['Customer ID','Avg VTAT','Avg CTAT','Cancelled Rides by Customer','Incomplete Rides','Incomplete Rides Reason'],axis = 1)

In [7]:
data.columns=data.columns.str.strip().str.lower().str.replace(' ','_')
data.columns

Index(['date', 'time', 'booking_id', 'booking_status', 'vehicle_type',
       'pickup_location', 'drop_location', 'reason_for_cancelling_by_customer',
       'cancelled_rides_by_driver', 'driver_cancellation_reason',
       'booking_value', 'ride_distance', 'driver_ratings', 'customer_rating',
       'payment_method'],
      dtype='object')

In [8]:
data['date_time'] = pd.to_datetime(data['date'] + ' ' + data['time'])

In [9]:
data.drop(['date','time'], axis = 1 , inplace = True)

In [10]:
cols = ['booking_value','ride_distance','driver_ratings','customer_rating']
for i in cols:
    data[i] = pd.to_numeric(data[i] , errors = 'coerce')

In [11]:
cols = ['booking_status','vehicle_type','pickup_location','drop_location','payment_method']
for i in cols:
    data[i] = data[i].astype('category')

In [12]:
col = data.pop('date_time')
data.insert(2,'date_time',col)

In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 14 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   booking_id                         150000 non-null  object        
 1   booking_status                     150000 non-null  category      
 2   date_time                          150000 non-null  datetime64[ns]
 3   vehicle_type                       150000 non-null  category      
 4   pickup_location                    150000 non-null  category      
 5   drop_location                      150000 non-null  category      
 6   reason_for_cancelling_by_customer  10500 non-null   object        
 7   cancelled_rides_by_driver          27000 non-null   float64       
 8   driver_cancellation_reason         27000 non-null   object        
 9   booking_value                      102000 non-null  float64       
 10  ride_distance       

In [15]:
table_name = 'uber'
data.to_sql(table_name, conn, if_exists = 'replace', index = False)
print(f"Data Successfully loaded into table : {table_name} into data base : {database}")

Data Successfully loaded into table : uber into data base : uber_db
